In [1]:
# LIBRERIAS

import numpy as np
import pandas as pd
import mlflow
import xgboost as xgb
import optuna

from sklearn.model_selection import cross_val_score, GridSearchCV
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score, log_loss

from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
import seaborn as sns

import plotly.express as px
import plotly.graph_objects as go

from tqdm import tqdm
from tqdm import TqdmWarning
from IPython.display import display
from typing import Sequence, cast
import os
import hashlib
import json
import tempfile
from pathlib import Path
import joblib

import warnings

warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

In [2]:
# CONFIGURACIÓN DE MLFLOW (SQLite con ruta absoluta respecto al cwd del kernel)

_mlflow_db_path = (Path.cwd() / "mlflow_experimentos.db").resolve()
_mlflow_sqlite_uri = "sqlite:///" + str(_mlflow_db_path).replace("\\", "/")
mlflow.set_tracking_uri(_mlflow_sqlite_uri)
mlflow.set_experiment("Championship_XGBoost_Model")
print(f"MLflow tracking DB (ruta absoluta): {_mlflow_db_path}")

MLflow tracking DB (ruta absoluta): /home/borjatortuero/Escritorio/Master/TFM/notebooks/mlflow_experimentos.db


In [3]:
# Directorio del notebook 
base_path = os.getcwd()
# Construir ruta relativa al dataset extendido
csv_path = os.path.join(base_path, '..', 'Data', 'Segunda_inglesa_data_extended.csv')
# Normalizar la ruta 
csv_path = os.path.abspath(csv_path)

# Cargamos el fichero
df = pd.read_csv(csv_path)

with open(csv_path, "rb") as _csv_f:
    CSV_DATA_SHA256 = hashlib.sha256(_csv_f.read()).hexdigest()

# Formateo estricto de fechas (crítico para las series temporales de XG)
df['Date'] = pd.to_datetime(df['Date'], dayfirst=True, errors='coerce')
df = df.sort_values(by=['Date']).reset_index(drop=True)

# 0: Victoria Local, 1: Victoria Visitante, 2: Empate
df['Target'] = np.select(
    [df['FTR'] == 'H', df['FTR'] == 'A', df['FTR'] == 'D'], 
    [0, 1, 2], default=-1
)

# Cálculo de probabilidades implícitas del mercado y overround (margen)
margin = (1/df['B365H']) + (1/df['B365D']) + (1/df['B365A'])
df['prob_B365H'] = (1/df['B365H']) / margin
df['prob_B365D'] = (1/df['B365D']) / margin
df['prob_B365A'] = (1/df['B365A']) / margin

# Expectativa de goles basada en el mercado (Over/Under 2.5)
df['goals_expectation'] = np.where(
    df['B365>2.5'].notna() & df['B365<2.5'].notna(),
    df['B365>2.5'] / (df['B365>2.5'] + df['B365<2.5']),
    0.5 
)

# Limpieza básica
df.dropna(subset=['B365H', 'B365D', 'B365A'], inplace=True)
df.dropna(axis=1, inplace=True)

display(df.head())

,Div,Date,Time,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,...,PCAHA,MaxCAHH,MaxCAHA,AvgCAHH,AvgCAHA,Target,prob_B365H,prob_B365D,prob_B365A,goals_expectation
0,E1,2021-08-06,19:45,Bournemouth,West Brom,2,2,D,1,1,...,1.80,2.16,1.83,2.09,1.77,2,0.412126,0.291658,0.296216,0.526316
1,E1,2021-08-07,15:00,Blackburn,Swansea,2,1,H,1,0,...,2.09,1.87,2.17,1.81,2.04,0,0.447368,0.276316,0.276316,0.588235
2,E1,2021-08-07,15:00,Bristol City,Blackpool,1,1,D,1,0,...,1.99,1.92,2.01,1.89,1.95,2,0.437854,0.276878,0.285268,0.569948
3,E1,2021-08-07,15:00,Cardiff,Barnsley,1,1,D,0,0,...,1.80,2.19,1.80,2.11,1.75,2,0.428346,0.277165,0.294488,0.588235
4,E1,2021-08-07,15:00,Derby,Huddersfield,1,1,D,1,1,...,2.01,1.90,2.07,1.87,1.98,2,0.207629,0.249155,0.543216,0.549738


In [4]:
# Nos quedamos con la primera temporada y mitad de la segunda para entrenar y con la mitad restante para predecir/validar
df_train = df.iloc[:-190].copy()
df_test = df.iloc[-190:].copy()

### **Modelo de Gradient Boosting (XGBoost)**
A diferencia de los modelos de Poisson (Maher/Dixon), XGBoost es un modelo algorítmico de árboles de decisión que no asume una distribución estadística previa para los goles. En su lugar, aprende patrones no lineales a partir de *features* dinámicas.

Para capturar la realidad temporal del fútbol, implementaremos un sistema de ingeniería de características basado en:
1. **Medias Móviles (K-Past):** Rendimiento en goles, tiros a puerta y córners en los últimos *k* partidos.
2. **Diferenciales de Forma:** Un sistema de puntuación dinámica ponderada (similar a sistemas Elo/Glicko) para ajustar el estado de forma de cada equipo jornada a jornada.
3. **Diferenciales de Rendimiento Acumulado:** Diferencias netas entre local y visitante para captar disparidades estructurales.

In [5]:
def calcular_features_dinamicas(df_base, k, gamma=0.33):
    """
    Calcula el estado de forma y las estadísticas rodantes históricas.
    """
    df_temp = df_base.copy()
    historial = []
    
    for index, row in df_temp.iterrows():
        pts_h = 3 if row['FTR'] == 'H' else (1 if row['FTR'] == 'D' else 0)
        pts_a = 3 if row['FTR'] == 'A' else (1 if row['FTR'] == 'D' else 0)
        
        historial.append({'Season': row.get('Season', 'All'), 'Date': row['Date'], 'Team': row['HomeTeam'], 'Role': 'Home', 
                          'GoalsScored': row['FTHG'], 'GoalsConceded': row['FTAG'], 
                          'ShotsTarget': row.get('HST', 0), 'Corners': row.get('HC', 0), 
                          'Points': pts_h, 'MatchID': index})
        historial.append({'Season': row.get('Season', 'All'), 'Date': row['Date'], 'Team': row['AwayTeam'], 'Role': 'Away', 
                          'GoalsScored': row['FTAG'], 'GoalsConceded': row['FTHG'], 
                          'ShotsTarget': row.get('AST', 0), 'Corners': row.get('AC', 0), 
                          'Points': pts_a, 'MatchID': index})
        
    df_h = pd.DataFrame(historial).sort_values(by=['Season', 'Team', 'Date'])
    
    # Estadísticas históricas (Rolling K)
    for m in ['GoalsScored', 'ShotsTarget', 'Corners']:
        df_h[f'Past_{k}_{m}'] = df_h.groupby(['Season', 'Team'])[m].transform(lambda x: x.shift(1).rolling(window=k).mean())
    
    df_h['Past_Streak'] = df_h.groupby(['Season', 'Team'])['Points'].transform(lambda x: x.shift(1).rolling(window=k).apply(lambda y: y.sum() / (3 * k), raw=True))
    pesos = np.arange(1, k + 1)
    df_h['Past_WStreak'] = df_h.groupby(['Season', 'Team'])['Points'].transform(lambda x: x.shift(1).rolling(window=k).apply(lambda y: np.average(y, weights=pesos) / 3, raw=True))
    
    df_h['Cum_Scored'] = df_h.groupby(['Season', 'Team'])['GoalsScored'].transform(lambda x: x.shift(1).cumsum().fillna(0))
    df_h['Cum_Conceded'] = df_h.groupby(['Season', 'Team'])['GoalsConceded'].transform(lambda x: x.shift(1).cumsum().fillna(0))
    df_h['GD'] = df_h['Cum_Scored'] - df_h['Cum_Conceded']
    
    cols_stats = [f'Past_{k}_GoalsScored', f'Past_{k}_ShotsTarget', f'Past_{k}_Corners', 'Past_Streak', 'Past_WStreak', 'GD']
    
    home_stats = df_h[df_h['Role'] == 'Home'].set_index('MatchID')[cols_stats]
    home_stats.columns = ['HGKPP', 'HSTKPP', 'HCKPP', 'HSt', 'HStWeighted', 'HTGD']
    away_stats = df_h[df_h['Role'] == 'Away'].set_index('MatchID')[cols_stats]
    away_stats.columns = ['AGKPP', 'ASTKPP', 'ACKPP', 'ASt', 'AStWeighted', 'ATGD']
    
    df_temp = df_temp.join(home_stats).join(away_stats)
    
    # Sistema de forma basado en decaimiento Gamma
    h_form_list, a_form_list = [], []
    estado_forma, temporada_actual = {}, None
    
    for index, row in df_temp.iterrows():
        if row.get('Season', 'All') != temporada_actual:
            estado_forma, temporada_actual = {}, row.get('Season', 'All')
            
        ht, at, res = row['HomeTeam'], row['AwayTeam'], row['FTR']
        if ht not in estado_forma: estado_forma[ht] = 1.0
        if at not in estado_forma: estado_forma[at] = 1.0
            
        f_ht, f_at = estado_forma[ht], estado_forma[at]
        h_form_list.append(f_ht)
        a_form_list.append(f_at)
        
        if res == 'H':
            estado_forma[ht], estado_forma[at] = f_ht + (gamma * f_at), f_at - (gamma * f_at)
        elif res == 'A':
            estado_forma[ht], estado_forma[at] = f_ht - (gamma * f_ht), f_at + (gamma * f_ht)
        else:
            diff = f_ht - f_at
            estado_forma[ht], estado_forma[at] = f_ht - (gamma * diff), f_at - (gamma * -diff)

    df_temp['HForm'], df_temp['AForm'] = h_form_list, a_form_list
    
    # Diferenciales finales (Features del modelo)
    df_temp['FormDifferential'] = df_temp['HForm'] - df_temp['AForm']
    df_temp['GDDifferential'] = df_temp['HTGD'] - df_temp['ATGD']
    df_temp['GKPP'] = df_temp['HGKPP'] - df_temp['AGKPP']
    df_temp['STKPP'] = df_temp['HSTKPP'] - df_temp['ASTKPP']
    df_temp['CKPP'] = df_temp['HCKPP'] - df_temp['ACKPP']
    df_temp['StDifferential'] = df_temp['HSt'] - df_temp['ASt']
    df_temp['StWeightedDifferential'] = df_temp['HStWeighted'] - df_temp['AStWeighted']
    
    return df_temp

In [6]:
# APLICACIÓN DE FEATURES Y PREPARACIÓN DE MATRICES
# -------------------------------------------------------------------------
k_optimo = 7  # Basado en experimentación previa

# Generamos las features para todo el dataset
df_processed = calcular_features_dinamicas(df, k=k_optimo)

features_modelo = [
    'FormDifferential', 'GDDifferential', 'GKPP', 'STKPP', 'CKPP', 
    'StDifferential', 'StWeightedDifferential',
    'prob_B365H', 'prob_B365D', 'prob_B365A', 'goals_expectation'
]

# Eliminamos los NaNs generados al principio de la serie por el Rolling K
df_clean = df_processed.dropna(subset=features_modelo + ['Target']).reset_index(drop=True)

# Volvemos a hacer el split Train/Test (asegurando el mismo bloque de 190 partidos finales)
train_data = df_clean.iloc[:-190].copy()
test_data = df_clean.iloc[-190:].copy()

X_train = train_data[features_modelo]
y_train = train_data['Target'].astype(int)

X_test = test_data[features_modelo]
y_test = test_data['Target'].astype(int)

In [7]:
# FUNCIÓN OBJETIVO (XGBoost Hyperparameter Tuning)
# -------------------------------------------------------------------------
def objective_xgboost(trial):
    """
    Función para Optuna. Evalúa combinaciones de hiperparámetros
    utilizando Validación Cruzada (CV) y retorna el Log Loss medio.
    """
    param = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'max_depth': trial.suggest_int('max_depth', 2, 6),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'gamma': trial.suggest_float('gamma', 0.01, 0.5),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.1, 2.0),
        'subsample': trial.suggest_float('subsample', 0.6, 0.9),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 0.9),
    }

    model = xgb.XGBClassifier(
        **param, 
        objective='multi:softprob', 
        num_class=3, 
        random_state=42, 
        n_jobs=-1
    )
    
    # Stratified K-Fold CV para robustez
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring='neg_log_loss')
    
    # Optuna minimiza por defecto, retornamos el log loss en positivo
    return -np.mean(scores)

In [8]:
# EJECUCIÓN DE LA OPTIMIZACIÓN (Optuna + MLflow)
# -------------------------------------------------------------------------

if mlflow.active_run() is not None:
    mlflow.end_run()

OPTUNA_SEED = 42

mlflow.start_run(run_name="XGBoost_Optuna_Optimization")
XG_RUN_ID = mlflow.active_run().info.run_id

mlflow.log_param("model_family", "xgboost")
mlflow.log_param("optimizer", "optuna")
mlflow.log_param("cv_folds", 5)
mlflow.log_param("k_rolling_window", k_optimo)
mlflow.log_param("data_csv_path", csv_path)
mlflow.log_param("data_csv_sha256", CSV_DATA_SHA256)
mlflow.log_param("df_train_rows", len(X_train))

print("⏳ Iniciando optimización XGBoost (Optuna)...\n")

# Ejecutamos el estudio de Optuna
study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=OPTUNA_SEED))
study.optimize(objective_xgboost, n_trials=30, show_progress_bar=True)

best_params = study.best_params
best_logloss = study.best_value

# Guardar hiperparámetros en MLflow
for key, value in best_params.items():
    mlflow.log_param(f"best_{key}", float(value) if isinstance(value, float) else value)
    
mlflow.log_metric("train_cv_logloss", best_logloss)
mlflow.log_metric("n_trials", len(study.trials))

# Guardar artefacto JSON con los hiperparámetros
_payload = {
    "k_rolling_window": k_optimo,
    "train_cv_logloss": best_logloss,
    "best_params": best_params
}
_td = tempfile.mkdtemp()
_json_path = os.path.join(_td, "xgboost_params.json")
with open(_json_path, "w", encoding="utf-8") as _jf:
    json.dump(_payload, _jf, indent=2)
mlflow.log_artifact(_json_path, artifact_path="model")

print("\n✅ ¡Optimización XGBoost Completada!")
print(f"🎯 Mejor Log-Loss (CV): {best_logloss:.6f}")
print("🧠 Mejores hiperparámetros:")
for k, v in best_params.items():
    print(f"   - {k}: {v}")

[I 2026-05-02 14:10:26,813] A new study created in memory with name: no-name-bfffe9d4-5c2b-416c-8429-541e22325998


⏳ Iniciando optimización XGBoost (Optuna)...



  0%|          | 0/30 [00:00<?, ?it/s]

[I 2026-05-02 14:10:37,113] Trial 0 finished with value: 1.0964067501919432 and parameters: {'n_estimators': 144, 'max_depth': 6, 'learning_rate': 0.05395030966670229, 'gamma': 0.30334265725654797, 'reg_lambda': 0.39643541684062933, 'subsample': 0.6467983561008608, 'colsample_bytree': 0.6174250836504598}. Best is trial 0 with value: 1.0964067501919432.
[I 2026-05-02 14:10:57,207] Trial 1 finished with value: 1.1114941862615628 and parameters: {'n_estimators': 267, 'max_depth': 5, 'learning_rate': 0.051059032093947576, 'gamma': 0.020086402204943198, 'reg_lambda': 1.9428287191077893, 'subsample': 0.8497327922401265, 'colsample_bytree': 0.6637017332034828}. Best is trial 0 with value: 1.0964067501919432.
[I 2026-05-02 14:10:57,907] Trial 2 finished with value: 1.0376336020704557 and parameters: {'n_estimators': 95, 'max_depth': 2, 'learning_rate': 0.02014847788415866, 'gamma': 0.26713065149979653, 'reg_lambda': 0.9206955354200199, 'subsample': 0.6873687420594126, 'colsample_bytree': 0.783

### **Predicción y Calibración en el Test Set**
A diferencia del modelo Dixon-Coles, que recalcula matemáticamente sus parámetros partido a partido (Rolling Window), XGBoost aprende reglas estructurales complejas sobre todo el conjunto de entrenamiento. 

Para garantizar que el modelo sea útil en *Value Betting*, es indispensable aplicar una **Calibración Isotónica**. Los modelos de árboles tienden a empujar las probabilidades hacia 0 o 1. La calibración ajusta estas probabilidades brutas para que reflejen fielmente la frecuencia real de los eventos (por ejemplo, asegurando que un evento predicho con un 40% de probabilidad ocurra realmente el 40% de las veces).

In [9]:
# ENTRENAMIENTO FINAL, CALIBRACIÓN Y PREDICCIÓN
# -------------------------------------------------------------------------

# 1. Instanciamos el modelo con los mejores parámetros de Optuna
modelo_final = xgb.XGBClassifier(
    **best_params, 
    objective='multi:softprob', 
    num_class=3, 
    random_state=42,
    n_jobs=-1
)

# 2. Entrenamos el modelo base
modelo_final.fit(X_train, y_train)

# 3. Calibración Isotónica (Crítico para apuestas de valor)
calibrated_final = CalibratedClassifierCV(modelo_final, method='isotonic', cv=5)
calibrated_final.fit(X_train, y_train)

# 4. Generamos predicciones sobre el Test Set (-190 partidos)
y_probs = calibrated_final.predict_proba(X_test)
y_preds = calibrated_final.predict(X_test)

# 5. Construimos el DataFrame de resultados con el mismo formato que Dixon
# Recordatorio XGBoost Output: 0 (Home), 1 (Away), 2 (Draw)
predictions_xg = pd.DataFrame({
    'Prob_Home': y_probs[:, 0],
    'Prob_Draw': y_probs[:, 2],
    'Prob_Away': y_probs[:, 1]
}, index=test_data.index)

odds_targets = ['B365H', 'B365D', 'B365A', 'PSH', 'PSD', 'PSA']
cols_to_keep = ['Date', 'HomeTeam', 'AwayTeam', 'FTHG', 'FTAG'] + [c for c in odds_targets if c in test_data.columns]

results_xg = pd.concat([test_data[cols_to_keep], predictions_xg], axis=1)

# Calculamos el Hit (Acierto del ganador)
outcome_conditions = [
    results_xg['FTHG'] > results_xg['FTAG'],
    results_xg['FTAG'] > results_xg['FTHG']
]
real_outcome = np.select(outcome_conditions, ['Prob_Home', 'Prob_Away'], default='Prob_Draw')
predicted_outcome = results_xg[['Prob_Home', 'Prob_Draw', 'Prob_Away']].idxmax(axis=1)

results_xg['Hit'] = (real_outcome == predicted_outcome).astype(int)

print("📊 Resultados de las últimas jornadas (XGBoost Calibrado):")
cols_show = ['Date', 'HomeTeam', 'AwayTeam', 'FTHG', 'FTAG', 'Prob_Home', 'Prob_Draw', 'Prob_Away', 'Hit']
display(results_xg[cols_show].head(10))

📊 Resultados de las últimas jornadas (XGBoost Calibrado):


,Date,HomeTeam,AwayTeam,FTHG,FTAG,Prob_Home,Prob_Draw,Prob_Away,Hit
1848,2025-02-05,Coventry,Leeds,0,2,0.186621,0.314446,0.498932,1
1849,2025-02-08,Sunderland,Watford,2,2,0.698227,0.167894,0.133879,0
1850,2025-02-08,West Brom,Sheffield Weds,2,1,0.459496,0.258168,0.282336,1
1851,2025-02-08,Norwich,Derby,1,1,0.549726,0.306396,0.143878,0
1852,2025-02-08,Sheffield United,Portsmouth,2,1,0.659995,0.273499,0.066506,1
1853,2025-02-09,Bristol City,Swansea,0,1,0.472446,0.345545,0.182009,0
1854,2025-02-11,Coventry,QPR,1,0,0.494264,0.281785,0.223951,1
1855,2025-02-11,Derby,Oxford,0,0,0.504523,0.235993,0.259484,0
1856,2025-02-11,Portsmouth,Cardiff,2,1,0.477426,0.215435,0.307139,1
1857,2025-02-11,Watford,Leeds,0,4,0.029331,0.477659,0.493010,1


### **Rendimiento del modelo y Evaluación Avanzada (Value Betting)**
Aplicamos exactamente el mismo pipeline de evaluación financiera y de clasificación estadística que en el modelo Dixon-Coles.
1. **Métricas de Machine Learning:** RPS (Ranked Probability Score), Log Loss y Accuracy.
2. **Grid Search Financiero:** Barrido tridimensional exhaustivo de hiperparámetros de apuesta (Margen, Fracción de Kelly, Cuota Máxima) para encontrar la frontera eficiente entre ROI y Balance.

In [10]:
def calculate_rps(probs, outcome_idx):
    """
    Calcula el Ranked Probability Score (RPS) para resultados 1X2.
    Estandarizado para coincidir con la evaluación de Dixon Coles.
    """
    # Ordenamos probs: [Home(0), Draw(2), Away(1)] -> [Home, Draw, Away]
    p_ordenadas = np.array([probs[0], probs[2], probs[1]])
    idx_ordenado = 0 if outcome_idx == 0 else (1 if outcome_idx == 2 else 2)
    
    cdf_p = np.cumsum(p_ordenadas)
    cdf_o = np.zeros(3)
    cdf_o[idx_ordenado:] = 1.0

    return np.sum((cdf_p[:-1] - cdf_o[:-1])**2) / (3 - 1)

In [11]:
# Función auxiliar para calcular Stake de Kelly
def get_kelly_stake(prob, odd, fraction):
    if odd <= 1:
        return 0
    b = odd - 1
    p = prob
    q = 1 - p
    f_star = (b * p - q) / b
    return max(0, f_star * fraction)

# --- EVALUACIÓN ESTADÍSTICA: XGBOOST ---
rps_list_xg = []
y_true_xg = []
y_pred_probs_xg = []
correct_preds_xg = 0

for idx, row in results_xg.iterrows():
    if row['FTHG'] > row['FTAG']:
        outcome = 0 # Home
    elif row['FTHG'] == row['FTAG']:
        outcome = 1 # Draw
    else:
        outcome = 2 # Away

    probs = [row['Prob_Home'], row['Prob_Draw'], row['Prob_Away']]
    y_true_xg.append(outcome)
    y_pred_probs_xg.append(probs)
    rps_list_xg.append(calculate_rps(probs, outcome))

    if np.argmax(probs) == outcome:
        correct_preds_xg += 1

avg_rps_xg = np.mean(rps_list_xg)
avg_logloss_xg = log_loss(y_true_xg, y_pred_probs_xg, labels=[0, 1, 2])
accuracy_xg = correct_preds_xg / len(results_xg) if len(results_xg) > 0 else 0.0

if mlflow.active_run() is None:
    raise RuntimeError("No hay run MLflow activo. Ejecuta la celda de Optuna primero.")

mlflow.log_metric("validation_accuracy", accuracy_xg)
mlflow.log_metric("validation_rps", avg_rps_xg)
mlflow.log_metric("validation_logloss", avg_logloss_xg)
mlflow.log_metric("validation_n_matches", len(results_xg))

print("-" * 60)
print("📉 MÉTRICAS DE CALIDAD DEL MODELO (XGBOOST)")
print("-" * 60)
print(f"🎯 Accuracy:     {accuracy_xg:.2%}")
print(f"🧩 RPS Promedio: {avg_rps_xg:.5f}")
print(f"🔥 Log Loss:     {avg_logloss_xg:.5f}")
print("-" * 60)

------------------------------------------------------------
📉 MÉTRICAS DE CALIDAD DEL MODELO (XGBOOST)
------------------------------------------------------------
🎯 Accuracy:     47.37%
🧩 RPS Promedio: 0.21870
🔥 Log Loss:     1.05229
------------------------------------------------------------


In [12]:
# PARÁMETROS Y PREPARACIÓN VECTORIZADA (GRID SEARCH FINANCIERO)
# ------------------------------------------------------
margins_to_test = np.round(np.arange(1.01, 2.005, 0.005), 3)
kellys_to_test = np.round(np.arange(0.01, 1.005, 0.005), 3)
max_odds_to_test = np.round(np.arange(2.5, 10.1, 0.1), 1)
MIN_ODD = 1.10
unit = 1
MIN_BETS_RATIO = 0.10

total_matches_xg = len(results_xg)
min_bets_threshold_xg = int(total_matches_xg * MIN_BETS_RATIO)

has_ps_xg = 'PSH' in results_xg.columns
n_bookies_xg = 2 if has_ps_xg else 1

print(f"📊 Total Partidos: {total_matches_xg} | Umbral Mínimo de Apuestas ({MIN_BETS_RATIO:.0%}): {min_bets_threshold_xg}")

outcomes_xg_arr = np.where(results_xg['FTHG'] > results_xg['FTAG'], 0,
                    np.where(results_xg['FTHG'] == results_xg['FTAG'], 1, 2))

# Aplanamiento de datos (Home, Draw, Away) x Bookies
probs_flat_xg = np.tile(np.concatenate([results_xg['Prob_Home'].values, results_xg['Prob_Draw'].values, results_xg['Prob_Away'].values]), n_bookies_xg)

if has_ps_xg:
    odds_flat_xg = np.concatenate([
        results_xg['B365H'].values, results_xg['B365D'].values, results_xg['B365A'].values,
        results_xg['PSH'].values, results_xg['PSD'].values, results_xg['PSA'].values
    ])
    bookies_labels_xg = np.concatenate([np.repeat('B365', total_matches_xg * 3), np.repeat('PS', total_matches_xg * 3)])
else:
    odds_flat_xg = np.concatenate([results_xg['B365H'].values, results_xg['B365D'].values, results_xg['B365A'].values])
    bookies_labels_xg = np.repeat('B365', total_matches_xg * 3)

won_flat_xg = np.tile(np.concatenate([outcomes_xg_arr == 0, outcomes_xg_arr == 1, outcomes_xg_arr == 2]), n_bookies_xg)
fechas_flat_xg = np.tile(np.concatenate([results_xg['Date'].values] * 3), n_bookies_xg)
partidos_flat_xg = np.tile(np.concatenate([(results_xg['HomeTeam'] + " vs " + results_xg['AwayTeam']).values] * 3), n_bookies_xg)
tipos_flat_xg = np.tile(np.concatenate([np.repeat('Home', total_matches_xg), np.repeat('Draw', total_matches_xg), np.repeat('Away', total_matches_xg)]), n_bookies_xg)

valid_mask_xg = ~np.isnan(odds_flat_xg) & ~np.isnan(probs_flat_xg) & (odds_flat_xg >= MIN_ODD)
b_flat_xg = odds_flat_xg - 1
with np.errstate(divide='ignore', invalid='ignore'):
    k_stake_base_xg = (probs_flat_xg * b_flat_xg - (1 - probs_flat_xg)) / b_flat_xg
    k_stake_base_xg = np.where(k_stake_base_xg > 0, k_stake_base_xg, 0)
profit_base_xg = np.where(won_flat_xg, k_stake_base_xg * b_flat_xg, -k_stake_base_xg)

# GRID SEARCH TRIDIMENSIONAL
grid_results_xg = []
best_roi_xg = -float('inf')
best_history_xg_df = pd.DataFrame()

for max_o in tqdm(max_odds_to_test, desc="🎯 Optimizando XG (Max Odd/Margen)"):
    mask_odd = valid_mask_xg & (odds_flat_xg <= max_o)

    for m in margins_to_test:
        mask_final = mask_odd & (probs_flat_xg * odds_flat_xg > m)
        n_bets = np.sum(mask_final)
        if n_bets < min_bets_threshold_xg:
            continue

        profit_m = np.sum(profit_base_xg[mask_final])
        balances = profit_m * kellys_to_test * unit
        rois = (balances / n_bets * 100)

        for i, k in enumerate(kellys_to_test):
            grid_results_xg.append({
                'Max_Odd': max_o, 'Margen': m, 'Kelly': k,
                'Apuestas': int(n_bets), 'Balance': balances[i], 'ROI(%)': rois[i]
            })

            if rois[i] > best_roi_xg:
                best_roi_xg = rois[i]
                best_params_xg = {'Margen': m, 'Kelly': k, 'Max_Odd': max_o}
                best_history_xg_df = pd.DataFrame({
                    'Date': fechas_flat_xg[mask_final], 'Match': partidos_flat_xg[mask_final], 'Bookie': bookies_labels_xg[mask_final],
                    'Type': tipos_flat_xg[mask_final], 'Odd': odds_flat_xg[mask_final], 'Stake': k_stake_base_xg[mask_final] * k,
                    'Result': np.where(won_flat_xg[mask_final], 'Win', 'Loss'), 'Ganancia': profit_base_xg[mask_final] * k * unit
                })

# RESULTADOS Y ANÁLISIS
df_grid_xg = pd.DataFrame(grid_results_xg)
df_grid_xg_sorted = df_grid_xg.sort_values(by=['ROI(%)', 'Balance'], ascending=False)

print("\n" + "=" * 60)
print(f"🏆 TOP 10 CONFIGURACIONES ENCONTRADAS (XGBOOST) (> {min_bets_threshold_xg} Apuestas)")
print("=" * 60)
display(df_grid_xg_sorted.head(10).reset_index(drop=True))

if not df_grid_xg_sorted.empty:
    best_row = df_grid_xg_sorted.iloc[0]
    print("\n" + "*" * 60)
    print("💎 MEJOR ESTRATEGIA SELECCIONADA (XGBOOST)")
    print("-" * 60)
    print(f"🔹 Max Odd:  {best_row['Max_Odd']}\n🔹 Margen:   {best_row['Margen']}\n🔹 Kelly:    {best_row['Kelly']}")
    print(f"🔹 ROI:      {best_row['ROI(%)']:.2f}%\n🔹 Balance:  {best_row['Balance']:.2f}")
    print("*" * 60 + "\n")

# Cierre de MLflow
if not df_grid_xg.empty:
    _br = df_grid_xg_sorted.iloc[0]
    mlflow.log_metric("betting_best_roi_pct", float(_br["ROI(%)"]))
    mlflow.log_metric("betting_best_balance", float(_br["Balance"]))
    mlflow.log_metric("betting_best_n_apuestas", int(_br["Apuestas"]))

mlflow.end_run()

📊 Total Partidos: 190 | Umbral Mínimo de Apuestas (10%): 19


🎯 Optimizando XG (Max Odd/Margen): 100%|██████████| 76/76 [00:02<00:00, 27.02it/s]



🏆 TOP 10 CONFIGURACIONES ENCONTRADAS (XGBOOST) (> 19 Apuestas)


,Max_Odd,Margen,Kelly,Apuestas,Balance,ROI(%)
0,5.0,1.155,0.01,83,-0.006146,-0.007405
1,5.0,1.160,0.01,75,-0.005605,-0.007474
2,4.8,1.155,0.01,79,-0.006211,-0.007862
3,5.1,1.155,0.01,84,-0.006614,-0.007874
4,5.1,1.160,0.01,76,-0.006074,-0.007992
5,4.7,1.155,0.01,75,-0.006253,-0.008338
6,4.8,1.160,0.01,72,-0.006061,-0.008418
7,4.9,1.155,0.01,80,-0.006838,-0.008548
8,4.4,1.155,0.01,71,-0.006234,-0.008780
9,5.2,1.155,0.01,87,-0.007691,-0.008840



************************************************************
💎 MEJOR ESTRATEGIA SELECCIONADA (XGBOOST)
------------------------------------------------------------
🔹 Max Odd:  5.0
🔹 Margen:   1.155
🔹 Kelly:    0.01
🔹 ROI:      -0.01%
🔹 Balance:  -0.01
************************************************************



In [13]:
# --- VISUALIZACIÓN DE LA MEJOR ESTRATEGIA: XGBOOST (ROI vs BALANCE) ---

if 'best_history_xg_df' in locals() and not df_grid_xg_sorted.empty:
    df_viz_roi_x = best_history_xg_df.sort_values('Date').reset_index(drop=True)
    df_viz_roi_x['Balance_Cum'] = df_viz_roi_x['Ganancia'].cumsum()
    df_viz_roi_x['MA_Balance'] = df_viz_roi_x['Balance_Cum'].rolling(window=7, min_periods=1).mean()

    roi_row_x = df_grid_xg_sorted.iloc[0]
    rx_val, rx_bal, rx_bets = roi_row_x['ROI(%)'], roi_row_x['Balance'], int(roi_row_x['Apuestas'])
    rx_marg, rx_kelly, rx_max_o = roi_row_x['Margen'], roi_row_x['Kelly'], roi_row_x['Max_Odd']

    bal_row_x = df_grid_xg_sorted.sort_values(by='Balance', ascending=False).iloc[0]

    if (bal_row_x['Margen'] != rx_marg) or (bal_row_x['Kelly'] != rx_kelly) or (bal_row_x['Max_Odd'] != rx_max_o):
        bx_m, bx_k, bx_mo = bal_row_x['Margen'], bal_row_x['Kelly'], bal_row_x['Max_Odd']
        mask_bal_x = valid_mask_xg & (odds_flat_xg <= bx_mo) & (probs_flat_xg * odds_flat_xg > bx_m)

        df_viz_bal_x = pd.DataFrame({
            'Date': fechas_flat_xg[mask_bal_x],
            'Ganancia': profit_base_xg[mask_bal_x] * bx_k * unit
        }).sort_values('Date').reset_index(drop=True)
    else:
        df_viz_bal_x = df_viz_roi_x.copy()

    df_viz_bal_x['Balance_Cum'] = df_viz_bal_x['Ganancia'].cumsum()
    df_viz_bal_x['MA_Balance'] = df_viz_bal_x['Balance_Cum'].rolling(window=7, min_periods=1).mean()

    bx_val, bx_bal_total, bx_bets = bal_row_x['ROI(%)'], bal_row_x['Balance'], int(bal_row_x['Apuestas'])
    bx_marg, bx_kelly, bx_max_o = bal_row_x['Margen'], bal_row_x['Kelly'], bal_row_x['Max_Odd']

    fig = go.Figure()

    fig.add_trace(go.Scatter(x=df_viz_bal_x.index, y=df_viz_bal_x['MA_Balance'], name='Tendencia (Max Balance)',
                             line=dict(color='rgba(255, 165, 0, 0.8)', width=2, dash='dot')))
    fig.add_trace(go.Scatter(x=df_viz_bal_x.index, y=df_viz_bal_x['Balance_Cum'],
                             name=f'<b>MAX BALANCE</b> ({bx_val:.2f}% ROI)',
                             line=dict(color='rgba(255, 215, 0, 0.8)', width=3)))

    fig.add_trace(go.Scatter(x=df_viz_roi_x.index, y=df_viz_roi_x['MA_Balance'], name='Tendencia (Max ROI)',
                             line=dict(color='rgba(255, 255, 255, 0.5)', width=2, dash='dot')))
    fig.add_trace(go.Scatter(x=df_viz_roi_x.index, y=df_viz_roi_x['Balance_Cum'],
                             name=f'<b>MAX ROI</b> ({rx_val:.2f}% ROI)',
                             line=dict(color='rgba(255, 255, 255, 0.9)', width=3)))

    subtitle_text = (f"<b>Config. ROI:</b> M: {rx_marg} | K: {rx_kelly} | MaxO: {rx_max_o} | Bets: {rx_bets}  "
                     f"      /       "
                     f"<b>Config. Balance:</b> M: {bx_marg} | K: {bx_kelly} | MaxO: {bx_max_o} | Bets: {bx_bets}")

    fig.update_layout(
        template='plotly_dark',
        title={'text': '<b>XGBOOST: Rendimiento Acumulado</b>', 'y': 1, 'x': 0.5, 'xanchor': 'center'},
        annotations=[dict(text=subtitle_text, showarrow=False, xref="paper", yref="paper", x=0.5, y=1.15, font=dict(size=11, color="gray"))],
        hovermode='x unified',
        xaxis=dict(title='Número de Apuesta'),
        yaxis=dict(title='Balance Acumulado (u)', zeroline=True, zerolinecolor='white'),
        legend=dict(orientation="h", yanchor="top", y=1.08, xanchor="center", x=0.5),
        margin=dict(t=100, b=50, l=50, r=50)
    )
    fig.show()
else:
    print("⚠️ No hay datos suficientes para graficar.")

In [14]:
model_dir = os.path.abspath(os.path.join(os.getcwd(), '..', 'src', 'ml_models'))
os.makedirs(model_dir, exist_ok=True)

model_path = os.path.join(model_dir, 'XGBoost_calibrated.joblib')

# Guardamos el modelo calibrado final (incluye el pipeline completo)
joblib.dump(calibrated_final, model_path)

print(f"✅ Modelo XGBoost guardado exitosamente en:\n{model_path}")

✅ Modelo XGBoost guardado exitosamente en:
/home/borjatortuero/Escritorio/Master/TFM/src/ml_models/XGBoost_calibrated.joblib
